In [ ]:
import numpy as np

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand':          ['E4', 'E5', 'E6', 'E7'],
    'Forearm':       ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm':           ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20'],
}
ELECTRODE_TO_REGION = {e: r for r, es in REGION_TO_ELECTRODES.items() for e in es}
REGION_TO_LABEL     = {'Middle_Finger': 0, 'Hand': 1, 'Forearm': 2, 'Arm': 3}
LABEL_TO_REGION     = {v: k for k, v in REGION_TO_LABEL.items()}


In [ ]:
from pathlib import Path

BIDS_ROOT   = Path("../../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"
subjects    = ["sub-p0002"]
session, task, space = "ses-01", "task-S1Map", "MNI152NLin2009cAsym"
n_runs, HRF_DELAY, WINDOW = 4, 6.0, 1

RESULTS_DIR = Path("results")
MODELS_DIR  = RESULTS_DIR / "models"
SHAP_DIR    = RESULTS_DIR / "shap"
FIGURES_DIR = SHAP_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import json

bold_json = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_bold.json"
with open(bold_json, 'r') as f:
    tr = float(json.load(f)["RepetitionTime"])
print(f"TR: {tr} s")


In [ ]:
from nilearn.image import load_img, index_img, new_img_like, resample_to_img
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.maskers import NiftiMasker

first_run_path = DERIVATIVES / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii"
first_run_img  = load_img(str(first_run_path))
ref_img        = index_img(first_run_img, 0)

atlas      = fetch_atlas_destrieux_2009()
s1_indices = [i for i, lab in enumerate(atlas.labels) if 'L G_postcentral' in str(lab) and i != 0]
atlas_img  = load_img(atlas.maps)
mask_data  = np.isin(atlas_img.get_fdata(), s1_indices).astype('uint8')
s1_mask    = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation='nearest')
masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
masker.fit(first_run_img)
print(f"Voxels in S1 mask: {int(np.sum(s1_mask_resampled.get_fdata() > 0))}")


In [ ]:
import pandas as pd

X_list, y_list = [], []
for run in range(1, n_runs + 1):
    events_path = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-{run}_events.tsv"
    df = pd.read_csv(events_path, sep='\t')
    stim = df[~df['trial_type'].isin(['Baseline', 'Jitter'])].copy()
    stim['region'] = stim['trial_type'].map(ELECTRODE_TO_REGION)

    bold_path = DERIVATIVES / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii"
    bold_img  = load_img(str(bold_path))
    for _, event in stim.iterrows():
        vol_idx = int(np.round((event["onset"] + HRF_DELAY) / tr))
        vols    = [vol_idx - WINDOW, vol_idx, vol_idx + WINDOW]
        if all(0 <= v < bold_img.shape[3] for v in vols):
            data = [masker.transform(index_img(bold_img, v)) for v in vols]
            data = [d[0] if len(d.shape) == 2 else d for d in data]
            X_list.append(np.mean(data, axis=0))
            y_list.append(event["region"])

X         = np.array(X_list)
y_encoded = np.array([REGION_TO_LABEL[r] for r in y_list])
print(f"X: {X.shape}, y: {y_encoded.shape}")


In [ ]:
mask_vol    = s1_mask_resampled.get_fdata()
xs, ys, zs  = np.where(mask_vol > 0)

x_min, x_max = int(xs.min()), int(xs.max()) + 1
y_min, y_max = int(ys.min()), int(ys.max()) + 1
z_min, z_max = int(zs.min()), int(zs.max()) + 1
bx, by, bz   = x_max - x_min, y_max - y_min, z_max - z_min

def to_3d(X_flat):
    """Reshape (n, n_voxels) flat array → (n, 2, bx, by, bz) 3D bounding box.
    Channel 0: signal; Channel 1: binary mask (constant 1 where real brain is).
    """
    out = np.zeros((len(X_flat), 2, bx, by, bz), dtype=np.float32)
    out[:, 0, xs - x_min, ys - y_min, zs - z_min] = X_flat
    out[:, 1, xs - x_min, ys - y_min, zs - z_min] = 1.0
    return out

print(f"S1 bounding box: ({bx}, {by}, {bz}) voxels")


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_all = StandardScaler()
X_scaled   = scaler_all.fit_transform(X)
X_3d_all   = to_3d(X_scaled)
print(f"X_3d_all shape: {X_3d_all.shape}")


In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Module, Conv3d, BatchNorm3d, MaxPool3d, AdaptiveAvgPool3d, Linear, Dropout

CHANNELS_1, CHANNELS_2, DROPOUT = 32, 64, 0.3

class SomatotopicCNN(Module):
    def __init__(self, ch1=CHANNELS_1, ch2=CHANNELS_2, dropout=DROPOUT, num_classes=4):
        super().__init__()
        self.conv1 = Conv3d(2, ch1, kernel_size=3, padding=1)
        self.bn1   = BatchNorm3d(ch1)
        self.pool1 = MaxPool3d(2)
        self.conv2 = Conv3d(ch1, ch2, kernel_size=3, padding=1, stride=2)
        self.bn2   = BatchNorm3d(ch2)
        self.gap    = AdaptiveAvgPool3d(1)
        self.drop   = Dropout(dropout)
        self.fc     = Linear(ch2, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.gap(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)

model = SomatotopicCNN()
model.load_state_dict(torch.load(MODELS_DIR / "best_cnn_model.pt", weights_only=True))
model.eval()
print("Model loaded.")


In [ ]:
import shap

X_tensor   = torch.FloatTensor(X_3d_all)
background = X_tensor[:20]

explainer  = shap.DeepExplainer(model, background)
# check_additivity=False: BatchNorm3d causes minor rounding violations in SHAP's
# additivity check — known limitation of DeepExplainer with BN layers.
shap_out   = explainer.shap_values(X_tensor, check_additivity=False)

# shap_out: list of 4 arrays, each shape (n, 2, bx, by, bz)
# Handle both SHAP API versions
if isinstance(shap_out, list):
    shap_values_3d = shap_out                          # list of 4 x (n, 2, bx, by, bz)
else:
    shap_values_3d = [shap_out[:, :, :, :, :, c] for c in range(4)]

print(f"Classes: {len(shap_values_3d)}, shape per class: {shap_values_3d[0].shape}")


In [ ]:
n_voxels   = X.shape[1]
xi         = xs - x_min
yi         = ys - y_min
zi         = zs - z_min

# Extract channel 0 (signal) from 3D SHAP values and project back to flat voxel space
shap_voxel = np.zeros((4, n_voxels))
for c in range(4):
    sv = shap_values_3d[c]                              # (n, 2, bx, by, bz)
    sv_ch0 = sv[:, 0, xi, yi, zi]                       # (n, n_voxels) — signal channel only
    shap_voxel[c] = np.mean(sv_ch0[y_encoded == c], axis=0)

shap_abs = np.abs(shap_voxel)
print(f"shap_voxel shape: {shap_voxel.shape}")


In [ ]:
from nilearn import plotting, datasets
from nilearn import surface as surf_utils
from IPython.display import display

for c in range(4):
    class_name = LABEL_TO_REGION[c]
    shap_img   = masker.inverse_transform(shap_abs[c].reshape(1, -1))
    nonzero    = shap_abs[c][shap_abs[c] > 0]
    thresh     = np.percentile(nonzero, 75) if len(nonzero) > 0 else 0.0

    view = plotting.view_img_on_surf(
        shap_img,
        surf_mesh='fsaverage',
        title=f'SHAP |importance|: {class_name}',
        cmap='hot',
        symmetric_cmap=False,
        threshold=thresh,
        vol_to_surf_kwargs={"interpolation": "nearest_most_frequent"}
    )
    view.save_as_html(FIGURES_DIR / f"shap_surf_{class_name}.html")
    display(view)


In [ ]:
import matplotlib.pyplot as plt

fsaverage = datasets.fetch_surf_fsaverage()

for c in range(4):
    class_name = LABEL_TO_REGION[c]
    shap_img   = masker.inverse_transform(shap_abs[c].reshape(1, -1))
    texture    = surf_utils.vol_to_surf(shap_img, fsaverage.pial_left)
    nonzero    = texture[texture > 0]
    vmax       = np.percentile(nonzero, 99) if len(nonzero) > 0 else 1.0
    thresh     = np.percentile(nonzero, 75) if len(nonzero) > 0 else 0.0

    for view_name in ('lateral', 'medial'):
        surf_fig = plotting.plot_surf_stat_map(
            fsaverage.infl_left, stat_map=texture,
            hemi='left', view=view_name,
            cmap='hot', threshold=thresh, colorbar=True,
            symmetric_cmap=False, vmax=vmax,
            bg_on_data=True, alpha=0.7,
            title=f'SHAP |{class_name}| — {view_name}',
            cbar_tick_format='%.2e'
        )
        surf_fig.savefig(str(FIGURES_DIR / f"shap_surf_{class_name}_{view_name}.png"),
                         dpi=150, bbox_inches='tight')
        plt.close(surf_fig)
    print(f"Saved: {class_name}")


In [ ]:
CLASS_ORDER = [LABEL_TO_REGION[c] for c in range(4)]
VIEWS       = ['lateral', 'medial']

fig, axes = plt.subplots(len(CLASS_ORDER), len(VIEWS), figsize=(10, 16))

for r, class_name in enumerate(CLASS_ORDER):
    for col, view_name in enumerate(VIEWS):
        ax       = axes[r, col]
        img_path = FIGURES_DIR / f"shap_surf_{class_name}_{view_name}.png"
        ax.imshow(plt.imread(str(img_path)))
        ax.axis('off')
        if r == 0:
            ax.set_title(view_name.capitalize(), fontsize=13, fontweight='bold', pad=6)
        if col == 0:
            ax.text(-0.04, 0.5, class_name.replace('_', ' '),
                    transform=ax.transAxes, fontsize=11,
                    va='center', ha='right', rotation=90)

fig.suptitle('CNN SHAP Absolute Importance — Left S1 per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_panel_all_classes.png", dpi=200, bbox_inches='tight')
plt.show()
plt.close()


In [ ]:
TOP_PERCENTILE = 75
winner_map     = np.argmax(shap_abs, axis=0)
max_shap       = np.amax(shap_abs, axis=0)
threshold      = np.percentile(max_shap, TOP_PERCENTILE)
winner_values  = np.where(max_shap >= threshold, winner_map + 1, 0).astype(np.float64)
winner_img     = masker.inverse_transform(winner_values.reshape(1, -1))

view = plotting.view_img_on_surf(
    winner_img,
    surf_mesh='fsaverage',
    title="CNN SHAP Winner-Take-All (1=Middle_Finger, 2=Hand, 3=Forearm, 4=Arm)",
    symmetric_cmap=False,
    vmin=1, vmax=4,
    threshold=1,
    vol_to_surf_kwargs={"interpolation": "nearest_most_frequent"}
)
view.save_as_html(FIGURES_DIR / "shap_winner_take_all.html")
display(view)

wta_texture = surf_utils.vol_to_surf(winner_img, fsaverage.pial_left)
wta_fig = plotting.plot_surf_stat_map(
    fsaverage.infl_left, stat_map=wta_texture,
    hemi='left', view='lateral',
    cmap='Set1', threshold=0.5, colorbar=True,
    symmetric_cmap=False, vmin=1, vmax=4,
    bg_on_data=True, alpha=0.7,
    title='CNN SHAP Winner-Take-All'
)
wta_fig.savefig(str(FIGURES_DIR / "shap_winner_take_all.png"), dpi=150, bbox_inches='tight')
plt.close(wta_fig)


In [ ]:
np.save(SHAP_DIR / "shap_values_3d.npy",   np.array(shap_values_3d, dtype=object), allow_pickle=True)
np.save(SHAP_DIR / "shap_voxel_maps.npy", shap_voxel)
print(f"Saved → {SHAP_DIR / 'shap_values_3d.npy'}")
print(f"Saved → {SHAP_DIR / 'shap_voxel_maps.npy'}")
